# DDRI cluster02 subset 재검증(Subset Recheck)

이 노트북은 `cluster02(주거 도착형)`에 대해 경량 subset을 재검증하고, 현재 2차 보강 결과보다 더 적은 피처로 성능을 유지할 수 있는지 확인하는 문서다.


## 1. 실험 목적(Experiment Goal)

- 담당 군집: `여기에 군집명 기입`
- 목표: 대표 대여소 3개를 대상으로 `2023 train / 2024 validation / 2025 test` 정책에 따라 공통 모델 실험을 수행한다.
- 필수 모델: `LinearRegression`, `LightGBM_RMSE`
- 필수 평가 지표: `RMSE`, `MAE`, `R²`


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

try:
    from lightgbm import LGBMRegressor
except ImportError:
    LGBMRegressor = None

BASE_DIR = Path('/Users/cheng80/Desktop/ddri_work/works/05_prediction_long')
TRAIN_PATH = Path('/Users/cheng80/Desktop/ddri_work/3조 공유폴더/대표대여소_예측데이터_15개/second_round_data/ddri_prediction_long_train_2023_2024_second_round_feature_collection.csv')
TEST_PATH = Path('/Users/cheng80/Desktop/ddri_work/3조 공유폴더/대표대여소_예측데이터_15개/second_round_data/ddri_prediction_long_test_2025_second_round_feature_collection.csv')
OUTPUT_DIR = BASE_DIR / 'output' / 'team_cluster_subset_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_STATION_GROUP = '주거 도착형'  # 담당 군집 고정
DISPLAY_NAME = TARGET_STATION_GROUP

RANDOM_STATE = 42


## 2. 입력 데이터 로드(Input Data Load)

- 공통 입력 정본은 `3조 공유폴더/대표대여소_예측데이터_15개/raw_data/` 아래 CSV 2개다.
- 개인이 별도 가공한 CSV를 정본처럼 사용하지 않는다.


In [2]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

print('train shape:', train_raw.shape)
print('test shape:', test_raw.shape)
print('station groups:', sorted(train_raw['station_group'].dropna().unique().tolist()))


train shape: (263160, 51)
test shape: (131400, 51)
station groups: ['생활권 혼합형', '아침 도착 업무 집중형', '업무/상업 혼합형', '외곽 주거형', '주거 도착형']


## 3. 담당 군집 필터링(Filter Assigned Cluster)

- 반드시 `station_group` 1개만 남긴다.
- 결과적으로 대표 대여소 3개만 남아야 한다.


In [3]:
train_group = train_raw.loc[train_raw['station_group'] == TARGET_STATION_GROUP].copy()
test_group = test_raw.loc[test_raw['station_group'] == TARGET_STATION_GROUP].copy()

print('train_group shape:', train_group.shape)
print('test_group shape:', test_group.shape)
print(train_group[['station_id', 'station_name']].drop_duplicates().sort_values('station_id').to_string(index=False))


train_group shape: (52632, 51)
test_group shape: (26280, 51)
 station_id  station_name
       2312  청담역 13번 출구 앞
       2354      청담역 2번출구
       4917    일원에코파크 주차장


## 4. 전처리 및 파생 피처 생성(Preprocessing And Feature Engineering)

공통 프로토콜:

- `date`는 날짜형으로 변환한다.
- `station_id`, `date`, `hour` 기준으로 정렬한다.
- `lag_1h`, `lag_24h`, `lag_168h`를 만든다.
- `rolling_mean_24h`, `rolling_std_24h`, `rolling_mean_168h`, `rolling_std_168h`를 만든다.
- rolling은 반드시 `station_id`별로 계산하고, `shift(1)` 후 rolling을 적용한다.
- `lag_168h`는 `1주 전 동일 요일·동일 시각 대여량`이다.


In [4]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    result['date'] = pd.to_datetime(result['date'])
    result = result.sort_values(['station_id', 'date', 'hour']).reset_index(drop=True)

    group = result.groupby('station_id', group_keys=False)['rental_count']
    result['lag_1h'] = group.shift(1)
    result['lag_24h'] = group.shift(24)
    result['lag_168h'] = group.shift(168)

    shifted = group.shift(1)
    result['rolling_mean_24h'] = shifted.rolling(24).mean().reset_index(level=0, drop=True)
    result['rolling_std_24h'] = shifted.rolling(24).std().reset_index(level=0, drop=True)
    result['rolling_mean_168h'] = shifted.rolling(168).mean().reset_index(level=0, drop=True)
    result['rolling_std_168h'] = shifted.rolling(168).std().reset_index(level=0, drop=True)
    return result

train_feat = build_features(train_group)
test_feat = build_features(test_group)

train_feat[['station_id', 'date', 'hour', 'rental_count', 'lag_1h', 'lag_24h', 'lag_168h']].head()


,station_id,date,hour,rental_count,lag_1h,lag_24h,lag_168h
0,2312,2023-01-01,0,0,NaN,NaN,NaN
1,2312,2023-01-01,1,0,0.0,NaN,NaN
2,2312,2023-01-01,2,0,0.0,NaN,NaN
3,2312,2023-01-01,3,0,0.0,NaN,NaN
4,2312,2023-01-01,4,0,0.0,NaN,NaN


## 5. 시간 분할 정책(Time-Based Split Policy)

- 학습(`Train`): `2023`
- 검증(`Validation`): `2024`
- 테스트(`Test`): `2025`

랜덤 분할은 금지한다.


In [5]:
train_2023 = train_feat.loc[train_feat['date'].dt.year == 2023].copy()
valid_2024 = train_feat.loc[train_feat['date'].dt.year == 2024].copy()
test_2025 = test_feat.copy()

print('train_2023:', train_2023.shape)
print('valid_2024:', valid_2024.shape)
print('test_2025:', test_2025.shape)


train_2023: (26280, 58)
valid_2024: (26352, 58)
test_2025: (26280, 58)


## 6. 입력 피처 목록(Input Feature List)

아래 목록은 공통 기본 피처다. 개인 실험에서 피처를 추가하거나 제거했다면 반드시 이유를 마크다운에 남긴다.


In [6]:
target_col = 'rental_count'

categorical_base_features = [
    'station_id',
    'cluster',
    'mapped_dong_code',
]

numeric_base_features = [
    'hour',
    'weekday',
    'month',
    'holiday',
    'temperature',
    'humidity',
    'precipitation',
    'wind_speed',
    'lag_1h',
    'lag_24h',
    'lag_168h',
    'rolling_mean_24h',
    'rolling_std_24h',
    'rolling_mean_168h',
    'rolling_std_168h',
    'hour_sin',
    'hour_cos',
]

subset_feature_map = {
    'baseline_full': [
        'is_weekend',
        'is_night_hour',
        'is_holiday_eve',
        'heavy_rain_flag',
        'station_elevation_m',
        'bus_stop_count_300m',
    ],
    'subset_a_living_pattern_core': [
        'is_weekend',
        'is_night_hour',
        'is_holiday_eve',
    ],
    'subset_b_living_pattern_rain': [
        'is_weekend',
        'is_night_hour',
        'is_holiday_eve',
        'heavy_rain_flag',
    ],
    'subset_c_living_pattern_location': [
        'is_weekend',
        'is_night_hour',
        'is_holiday_eve',
        'station_elevation_m',
        'bus_stop_count_300m',
    ],
    'subset_d_current_compact_best': [
        'is_weekend',
        'is_night_hour',
        'is_holiday_eve',
        'heavy_rain_flag',
        'station_elevation_m',
        'bus_stop_count_300m',
    ],
}

def get_feature_columns(subset_name: str):
    numeric_features = numeric_base_features + subset_feature_map[subset_name]
    feature_cols = categorical_base_features + numeric_features
    return feature_cols, categorical_base_features, numeric_features

pd.DataFrame([
    {
        'subset_name': subset_name,
        'feature_count': len(get_feature_columns(subset_name)[0]),
        'extra_feature_count': len(extra_features),
        'extra_features': ', '.join(extra_features),
    }
    for subset_name, extra_features in subset_feature_map.items()
])

,subset_name,feature_count,extra_feature_count,extra_features
0,baseline_full,26,6,"is_weekend, is_night_hour, is_holiday_eve, hea..."
1,subset_a_living_pattern_core,23,3,"is_weekend, is_night_hour, is_holiday_eve"
2,subset_b_living_pattern_rain,24,4,"is_weekend, is_night_hour, is_holiday_eve, hea..."
3,subset_c_living_pattern_location,25,5,"is_weekend, is_night_hour, is_holiday_eve, sta..."
4,subset_d_current_compact_best,26,6,"is_weekend, is_night_hour, is_holiday_eve, hea..."


## 7. 평가 함수(Evaluation Function)

모든 팀원은 아래 3개 점수를 동일하게 기록한다.

- `RMSE`
- `MAE`
- `R²`


In [7]:
def evaluate_regression(y_true, y_pred):
    return {
        'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'r2': float(r2_score(y_true, y_pred)),
    }

def build_linear_pipeline():
    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
            ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric_features),
        ]
    )
    return Pipeline([
        ('preprocessor', preprocessor),
        ('model', LinearRegression()),
    ])

def build_lgbm_model(objective='regression'):
    if LGBMRegressor is None:
        raise ImportError('lightgbm 패키지가 설치되어 있어야 합니다.')
    return LGBMRegressor(
        objective=objective,
        n_estimators=400,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
    )

## 8. subset별 모델 재검증(Model Recheck By Subset)

이번 노트북은 `LightGBM_RMSE`를 모든 subset에서 비교하고,
`LightGBM_Poisson`은 현재 compact 후보(`subset_d_current_compact_best`)에서만 참고용으로 1회 확인한다.

validation 결과를 기준으로 우세 `subset + 모델` 조합을 선택한다.

In [8]:
results = []

y_train = train_2023[target_col]
y_valid = valid_2024[target_col]

for subset_name in subset_feature_map:
    feature_cols, categorical_features, numeric_features = get_feature_columns(subset_name)
    X_train = train_2023[feature_cols]
    X_valid = valid_2024[feature_cols]

    if LGBMRegressor is not None:
        lgbm_model = build_lgbm_model(objective='regression')
        lgbm_model.fit(X_train, y_train)
        lgbm_pred = lgbm_model.predict(X_valid)
        lgbm_metrics = evaluate_regression(y_valid, lgbm_pred)
        results.append({
            'station_group': TARGET_STATION_GROUP,
            'subset_name': subset_name,
            'feature_count': len(feature_cols),
            'model': 'LightGBM_RMSE',
            'split': 'validation_2024',
            **lgbm_metrics,
        })

        if subset_name == 'subset_d_current_compact_best':
            poisson_model = build_lgbm_model(objective='poisson')
            poisson_model.fit(X_train, y_train)
            poisson_pred = poisson_model.predict(X_valid)
            poisson_metrics = evaluate_regression(y_valid, poisson_pred)
            results.append({
                'station_group': TARGET_STATION_GROUP,
                'subset_name': subset_name,
                'feature_count': len(feature_cols),
                'model': 'LightGBM_Poisson',
                'split': 'validation_2024',
                **poisson_metrics,
            })

validation_results = pd.DataFrame(results).sort_values(['rmse', 'mae']).reset_index(drop=True)
validation_results

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001102 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1605
[LightGBM] [Info] Number of data points in the train set: 26280, number of used features: 25
[LightGBM] [Info] Start training from score 0.450038


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000931 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1596
[LightGBM] [Info] Number of data points in the train set: 26280, number of used features: 22
[LightGBM] [Info] Start training from score 0.450038


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001075 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1598
[LightGBM] [Info] Number of data points in the train set: 26280, number of used features: 23
[LightGBM] [Info] Start training from score 0.450038


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000966 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1603
[LightGBM] [Info] Number of data points in the train set: 26280, number of used features: 24
[LightGBM] [Info] Start training from score 0.450038


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000945 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1605
[LightGBM] [Info] Number of data points in the train set: 26280, number of used features: 25
[LightGBM] [Info] Start training from score 0.450038


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000932 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1605
[LightGBM] [Info] Number of data points in the train set: 26280, number of used features: 25
[LightGBM] [Info] Start training from score -0.798423


,station_group,subset_name,feature_count,model,split,rmse,mae,r2
0,주거 도착형,subset_d_current_compact_best,26,LightGBM_Poisson,validation_2024,0.810864,0.501680,0.410771
1,주거 도착형,subset_c_living_pattern_location,25,LightGBM_RMSE,validation_2024,0.820785,0.510339,0.396265
2,주거 도착형,subset_b_living_pattern_rain,24,LightGBM_RMSE,validation_2024,0.821178,0.511459,0.395687
3,주거 도착형,subset_a_living_pattern_core,23,LightGBM_RMSE,validation_2024,0.825207,0.511743,0.389743
4,주거 도착형,baseline_full,26,LightGBM_RMSE,validation_2024,0.826075,0.512987,0.388458
5,주거 도착형,subset_d_current_compact_best,26,LightGBM_RMSE,validation_2024,0.826075,0.512987,0.388458


## 9. 우세 subset 및 모델 선택 후 재학습(Refit Best Subset Model)

- validation 결과를 기준으로 우세 `subset + 모델` 조합을 선택한다.
- 선택 조합을 `2023 + 2024`로 재학습한다.
- 그 다음 `2025` 테스트셋 점수를 산출한다.

In [9]:
best_row = validation_results.sort_values(['rmse', 'mae']).iloc[0]
best_model_name = best_row['model']
best_subset_name = best_row['subset_name']
best_feature_cols, categorical_features, numeric_features = get_feature_columns(best_subset_name)
print('best_model_name:', best_model_name)
print('best_subset_name:', best_subset_name)

refit_train = train_feat.copy()
X_refit = refit_train[best_feature_cols]
y_refit = refit_train[target_col]
X_test = test_2025[best_feature_cols]
y_test = test_2025[target_col]

if best_model_name == 'LightGBM_Poisson':
    best_model = build_lgbm_model(objective='poisson')
else:
    best_model = build_lgbm_model(objective='regression')

best_model.fit(X_refit, y_refit)
test_pred = best_model.predict(X_test)
test_metrics = evaluate_regression(y_test, test_pred)

test_results = pd.DataFrame([{ 
    'station_group': TARGET_STATION_GROUP,
    'subset_name': best_subset_name,
    'feature_count': len(best_feature_cols),
    'model': best_model_name,
    'split': 'test_2025_refit',
    **test_metrics,
}])
test_results

best_model_name: LightGBM_Poisson
best_subset_name: subset_d_current_compact_best
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001371 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1686
[LightGBM] [Info] Number of data points in the train set: 52632, number of used features: 25
[LightGBM] [Info] Start training from score -0.725391


,station_group,subset_name,feature_count,model,split,rmse,mae,r2
0,주거 도착형,subset_d_current_compact_best,26,LightGBM_Poisson,test_2025_refit,0.798959,0.499701,0.510766


## 10. 결과 저장(Result Save)

- 결과 CSV는 담당 군집명이 보이게 저장한다.
- 최소 2개 split(`validation_2024`, `test_2025_refit`)이 모두 있어야 한다.


In [10]:
final_metrics = pd.concat([validation_results, test_results], ignore_index=True)
safe_group_name = TARGET_STATION_GROUP.replace('/', '_').replace(' ', '_')
metrics_path = OUTPUT_DIR / f'ddri_{safe_group_name}_subset_recheck_model_metrics.csv'
final_metrics.to_csv(metrics_path, index=False)
print('saved:', metrics_path)
final_metrics


saved: /Users/cheng80/Desktop/ddri_work/works/05_prediction_long/output/team_cluster_subset_outputs/ddri_주거_도착형_subset_recheck_model_metrics.csv


,station_group,subset_name,feature_count,model,split,rmse,mae,r2
0,주거 도착형,subset_d_current_compact_best,26,LightGBM_Poisson,validation_2024,0.810864,0.501680,0.410771
1,주거 도착형,subset_c_living_pattern_location,25,LightGBM_RMSE,validation_2024,0.820785,0.510339,0.396265
2,주거 도착형,subset_b_living_pattern_rain,24,LightGBM_RMSE,validation_2024,0.821178,0.511459,0.395687
3,주거 도착형,subset_a_living_pattern_core,23,LightGBM_RMSE,validation_2024,0.825207,0.511743,0.389743
4,주거 도착형,baseline_full,26,LightGBM_RMSE,validation_2024,0.826075,0.512987,0.388458
5,주거 도착형,subset_d_current_compact_best,26,LightGBM_RMSE,validation_2024,0.826075,0.512987,0.388458
6,주거 도착형,subset_d_current_compact_best,26,LightGBM_Poisson,test_2025_refit,0.798959,0.499701,0.510766


## 11. 결과 해석(Result Interpretation)

아래 항목을 팀원별로 반드시 직접 작성한다.

- validation 기준 우세 모델은 무엇인가
- test 점수는 어느 정도인가
- 이 군집에서 예측이 어려운 시간대가 보이는가
- 날씨나 1주 전 동일 시각(`lag_168h`) 영향이 큰 것처럼 보이는가
- 다음 실험에서 추가할 피처 또는 제거할 피처는 무엇인가


## 12. 제출 전 체크리스트(Submission Checklist)

- [ ] 담당 군집 3개 대여소만 사용했는가
- [ ] `2023 train / 2024 validation / 2025 test`를 지켰는가
- [ ] `LinearRegression`과 `LightGBM_RMSE`를 모두 돌렸는가
- [ ] `RMSE`, `MAE`, `R²`를 validation/test 모두 기록했는가
- [ ] lag/rolling을 `station_id`별로 만들었는가
- [ ] 테스트셋으로 모델 선택을 하지 않았는가
- [ ] 노트북 마크다운에 전처리와 해석이 남아 있는가
- [ ] 차트와 표기가 `한글(영문)` 형식인가
